# 모델 저장과 활용

**이 노트북에서 할 것**

1. 학습과 예측은 성격이 다르다
2. `joblib` 으로 저장하고 불러오기
3. ★ 모델만 저장하면 조용히 틀린다
4. Pipeline 통째로 저장하기
5. ★ 피처 순서가 위험한 이유
6. 함께 남겨야 할 것

---

> 💡 **짧은 노트북입니다**
>
> 새로운 알고리즘은 없습니다. **지금까지 만든 모델을 어떻게 남기고 다시 쓰는가**만 다룹니다.
> 코드 자체는 두 줄이고, 나머지는 **무엇을 함께 저장해야 하는가**에 대한 이야기입니다.

In [ ]:
import os
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from _style import setup
from _ml import build_dataset, time_split, FEATURES, SEED

setup()
pd.set_option("display.width", 130)

# 교안 22에서 쓰던 것과 같은 데이터입니다.
df = build_dataset()
train, test, _ = time_split(df)

tr = train.sample(8000, random_state=SEED)
te = test.sample(2000, random_state=SEED)

X_train, y_train = tr[FEATURES], tr["target_up"]
X_test, y_test = te[FEATURES], te["target_up"]

print(f"학습 {len(tr):,}행 / 테스트 {len(te):,}행 / 피처 {len(FEATURES)}개")

---
## 1. 학습과 예측은 성격이 다릅니다

| | 학습 | 예측 |
|---|---|---|
| 소요 시간 | 몇 분 ~ 몇 시간 | **밀리초** |
| 빈도 | 주 1회, 월 1회 | **초당 수백 번** |
| 필요한 것 | 전체 학습 데이터 | 모델 + 입력 하나 |

**매번 다시 학습할 수는 없습니다.** 얼마나 차이 나는지 직접 재봅니다.

In [ ]:
# 전처리 + 모델을 Pipeline 으로 묶습니다 (교안 21에서 배운 것)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=100, max_depth=8,
                                     random_state=SEED, n_jobs=-1)),
])

# 학습 시간 측정
t0 = time.perf_counter()
pipe.fit(X_train, y_train)
학습시간 = time.perf_counter() - t0

# 예측 시간 측정 (한 건)
한건 = X_test.head(1)
pipe.predict(한건)                      # 워밍업
t0 = time.perf_counter()
for _ in range(50):
    pipe.predict(한건)
예측시간 = (time.perf_counter() - t0) / 50

print(f"학습 1회  {학습시간 * 1000:>9.1f} ms")
print(f"예측 1건  {예측시간 * 1000:>9.1f} ms")
print(f"차이      {학습시간 / 예측시간:>9.0f}배")
print()
print(f"정확도 {pipe.score(X_test, y_test):.4f}")

> 💡 **사용자가 예측을 요청할 때마다 학습한다면**
>
> 8,000행으로도 몇백 배 차이입니다. 90,000행 전체라면 응답이 몇 분씩 걸립니다.
>
> **학습은 미리 해두고, 결과만 꺼내 쓰는 것**이 저장의 이유입니다.

---
## 2. joblib 으로 저장하고 불러오기

```python
joblib.dump(model, "model.pkl")      # 저장
model = joblib.load("model.pkl")     # 불러오기
```

**두 줄이 전부입니다.**

In [ ]:
os.makedirs("models", exist_ok=True)

# ── 저장 ──
joblib.dump(pipe, "models/pipeline.pkl")

크기 = os.path.getsize("models/pipeline.pkl") / 1024
print(f"저장 완료 : models/pipeline.pkl  ({크기:,.0f} KB)")

# ── 불러오기 ──
loaded = joblib.load("models/pipeline.pkl")

# 원래 모델과 같은 예측을 하는지 확인합니다
원본예측 = pipe.predict(X_test)
로드예측 = loaded.predict(X_test)

print(f"불러오기 완료")
print(f"예측이 완전히 같은가? {np.array_equal(원본예측, 로드예측)}")

In [ ]:
# 불러오는 데 걸리는 시간도 재봅니다.
t0 = time.perf_counter()
for _ in range(20):
    joblib.load("models/pipeline.pkl")
로드시간 = (time.perf_counter() - t0) / 20

print(f"로드 1회  {로드시간 * 1000:>8.1f} ms")
print(f"예측 1건  {예측시간 * 1000:>8.1f} ms")
print()
print(f"로드가 예측보다 {로드시간 / 예측시간:.1f}배 느립니다.")

> ⚠️ **요청마다 모델을 로드하면 안 됩니다**
>
> 파일을 읽는 것도 비용입니다. 예측보다 로드가 더 오래 걸립니다.
> **서버가 뜰 때 한 번 로드**하고 메모리에 들고 있어야 합니다.
>
> 교안 17의 커넥션 풀과 같은 발상입니다. 비싼 것은 미리 준비해 두고 재사용합니다.

> 💡 **`pickle` 대신 `joblib` 을 쓰는 이유**
>
> 둘 다 파이썬 객체를 파일로 만듭니다. 다만 `joblib` 은 **큰 Numpy 배열을 더 효율적으로** 저장합니다.
> 머신러닝 모델은 내부가 대부분 배열이라 `joblib` 이 유리합니다.

> ⚠️ **출처를 모르는 pkl 파일을 열지 마세요**
>
> `pickle` 형식은 **불러올 때 코드가 실행될 수 있는** 구조입니다.
> 사내에서 만든 것, 신뢰할 수 있는 출처의 것만 쓰세요.

---
## 3. ★ 모델만 저장하면 조용히 틀립니다

```
학습할 때    원본 데이터 → StandardScaler → 모델
예측할 때    새 데이터   →     ???       → 모델
```

**스케일러가 없으면 새 데이터를 어떤 기준으로 변환할지 알 수 없습니다.**

직접 재현해 봅니다.

In [ ]:
# 전처리기와 모델을 따로 두고, 모델만 저장했다고 가정합니다.

scaler = StandardScaler().fit(X_train)
model = RandomForestClassifier(n_estimators=100, max_depth=8,
                               random_state=SEED, n_jobs=-1)
model.fit(scaler.transform(X_train), y_train)

# ⭕ 변환하고 예측
정상 = model.predict(scaler.transform(X_test))

# ❌ 변환을 빠뜨리고 예측 (스케일러를 저장 안 했다면 이렇게 됩니다)
#    .values 로 넘기는 이유는 sklearn 의 열 이름 경고를 피하기 위해서입니다.
#    실제로는 이 경고조차 없는 경우가 많습니다.
누락 = model.predict(X_test.values)

print(f"{'':<20}{'정확도':>10}")
print("-" * 32)
print(f"{'변환하고 예측':<19}{accuracy_score(y_test, 정상):>10.4f}")
print(f"{'변환을 빠뜨리면':<18}{accuracy_score(y_test, 누락):>10.4f}")
print()
print(f"예측이 달라진 건수 : {(정상 != 누락).sum():,} / {len(정상):,}  "
      f"({(정상 != 누락).mean():.1%})")

> ⚠️ **에러가 나지 않습니다**
>
> 예측 결과의 **38%가 달라졌는데** 아무 경고도 없습니다.
> 정확도만 보면 오히려 조금 높게 나오기도 합니다. 우연입니다.
>
> **동작은 하는데 결과가 틀린** 유형의 문제입니다. 교안 21의 데이터 누수와 같은 성격입니다.

> 💡 **인코더도 마찬가지입니다**
>
> `전기전자 → 3` 이라는 대응이 인코더 안에 들어 있습니다.
> 인코더를 잃어버리면 그 대응을 복원할 방법이 없습니다.

---
## 4. 해결 — Pipeline 을 통째로 저장

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier()),
])
joblib.dump(pipe, "pipeline.pkl")
```

**전처리와 모델이 한 덩어리로 저장**됩니다. 불러오면 `predict` 한 번에 스케일링까지 자동으로 적용됩니다.

In [ ]:
# 앞에서 저장한 Pipeline 을 다시 불러와 확인합니다.
loaded = joblib.load("models/pipeline.pkl")

print("Pipeline 안에 들어 있는 것:")
for 이름, 단계 in loaded.steps:
    print(f"  {이름:<10} {type(단계).__name__}")

print()
# 스케일러가 학습 때 계산한 평균이 그대로 들어 있습니다
평균 = loaded.named_steps["scaler"].mean_
print(f"스케일러가 기억하고 있는 평균 (앞 3개) : {np.round(평균[:3], 4)}")
print(f"학습 데이터의 실제 평균     (앞 3개) : {np.round(X_train.mean().values[:3], 4)}")
print()
print("변환 기준이 파일 안에 함께 저장되어 있습니다.")

In [ ]:
# 원본 데이터를 그대로 넣어도 알아서 변환됩니다.
새데이터 = X_test.head(3)

print("입력 (원본 스케일):")
print(새데이터.round(4).to_string(index=False))
print()
print(f"예측 : {loaded.predict(새데이터)}")
print(f"확률 : {np.round(loaded.predict_proba(새데이터)[:, 1], 4)}")
print()
print("스케일링을 직접 하지 않았는데도 정상 동작합니다.")

---
## 5. ★ 피처 순서가 위험합니다

모델은 열 이름을 모릅니다. **위치로만** 판단합니다.

거래량과 등락률의 순서가 바뀌면 어떻게 될까요?

In [ ]:
뒤집기 = X_test[FEATURES[::-1]]          # 열 순서를 거꾸로

print(f"원래 순서 : {FEATURES[:3]} ...")
print(f"뒤집은 순서 : {FEATURES[::-1][:3]} ...")
print()

# ① DataFrame 으로 넘기면 — sklearn 이 열 이름을 확인합니다
try:
    loaded.predict(뒤집기)
    print("① DataFrame : 통과")
except Exception as e:
    print(f"① DataFrame : {type(e).__name__}")
    print(f"   {str(e).split(chr(10))[0]}")
    print("   → 열 이름이 있어서 sklearn 이 막아줍니다")

In [ ]:
# ② numpy 배열로 넘기면 — 열 이름이 사라집니다
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")        # 경고만 끄고 실행
    정상예측 = loaded.predict(X_test.values)
    뒤집힌예측 = loaded.predict(뒤집기.values)

차이 = (정상예측 != 뒤집힌예측).sum()
print(f"② numpy 배열 : 에러 없이 실행됨")
print(f"   예측이 달라진 건수 : {차이:,} / {len(정상예측):,}  ({차이 / len(정상예측):.1%})")
print()
print("   → 열 이름이 없으니 순서를 확인할 방법이 없습니다. 조용히 틀립니다.")

> ⚠️ **numpy 배열로 넘기면 sklearn 도 막아주지 못합니다**
>
> DataFrame 을 쓰면 최신 sklearn 이 열 이름을 비교해 에러를 냅니다.
> 그런데 `.values` 로 배열을 만들어 넘기면 **이름이 사라져서 검사가 불가능**합니다.
>
> 웹 서버에서 JSON 을 받아 배열로 만들어 넘기는 경우가 흔합니다. 실무에서 자주 생기는 사고입니다.

> 💡 **그래서 피처 목록을 함께 저장합니다**
>
> 예측 전에 저장된 순서대로 열을 정렬하면 이 문제가 사라집니다.

In [ ]:
# 안전하게 예측하는 함수
def 안전한_예측(bundle, 입력_df):
    """
    저장된 피처 목록 순서대로 열을 맞춘 뒤 예측한다.

    ★ 이 한 줄이 피처 순서 사고를 막습니다.
    """
    정렬된 = 입력_df[bundle["features"]]        # 저장된 순서로 재배열
    return bundle["pipeline"].predict(정렬된)


# 번들을 만들어 테스트
테스트번들 = {"pipeline": loaded, "features": FEATURES}

결과1 = 안전한_예측(테스트번들, X_test)
결과2 = 안전한_예측(테스트번들, 뒤집기)        # 뒤집힌 것을 넣어도

print(f"뒤집힌 입력으로 예측해도 같은 결과인가? {np.array_equal(결과1, 결과2)}")
print()
print("열 순서가 어떻든 저장된 순서로 맞춰주므로 안전합니다.")

---
## 6. 함께 남겨야 할 것

파일 하나만 덜렁 두면 **몇 달 뒤에 아무도 못 씁니다.**

| 남길 것 | 이유 |
|---|---|
| **전처리기 (Pipeline)** | 같은 변환을 적용해야 함 |
| **피처 목록과 순서** | 순서가 바뀌면 엉뚱한 예측 |
| 학습 데이터 기간 | 언제까지의 데이터로 배웠나 |
| 성능 지표 | 나중에 비교할 기준 |
| 라이브러리 버전 | 버전이 다르면 로드 실패 가능 |

In [ ]:
import sklearn

# 모델과 메타데이터를 한 덩어리로 묶습니다.
번들 = {
    "pipeline": pipe,                                   # 전처리 + 모델
    "features": FEATURES,                               # 피처 목록과 순서
    "train_period": (str(tr["date"].min().date()),
                     str(tr["date"].max().date())),     # 학습 기간
    "test_accuracy": float(pipe.score(X_test, y_test)), # 성능 지표
    "sklearn_version": sklearn.__version__,             # 라이브러리 버전
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

joblib.dump(번들, "models/model_bundle.pkl")
print(f"저장 : models/model_bundle.pkl  "
      f"({os.path.getsize('models/model_bundle.pkl') / 1024:,.0f} KB)")

In [ ]:
# 몇 달 뒤에 이 파일만 받았다고 상상해 보세요.
b = joblib.load("models/model_bundle.pkl")

print("이 모델에 대해 알 수 있는 것:")
print(f"  학습 기간   {b['train_period'][0]} ~ {b['train_period'][1]}")
print(f"  테스트 정확도 {b['test_accuracy']:.4f}")
print(f"  sklearn     {b['sklearn_version']}  (현재 {sklearn.__version__})")
print(f"  생성 시각    {b['created_at']}")
print(f"  피처 {len(b['features'])}개 : {b['features'][:3]} ...")
print()
print(f"예측 : {안전한_예측(b, X_test.head(5))}")

> 💡 **JSON 으로 따로 남기는 방법도 있습니다**
>
> `pkl` 은 파이썬으로만 열 수 있습니다. 메타데이터는 사람이 읽을 수 있게 두는 편이 낫습니다.
>
> ```
> models/
>   pipeline.pkl      모델 (joblib)
>   meta.json         피처 목록, 학습 기간, 성능  ← 텍스트 에디터로 열림
> ```

In [ ]:
import json

# 메타데이터만 따로 JSON 으로도 남겨둡니다.
메타 = {k: v for k, v in 번들.items() if k != "pipeline"}

with open("models/meta.json", "w", encoding="utf-8") as f:
    json.dump(메타, f, ensure_ascii=False, indent=2)

print("models/meta.json")
print(open("models/meta.json", encoding="utf-8").read())

---
## 7. 모델도 낡습니다

> 💡 **데이터 드리프트(data drift)**
>
> 시장 상황이 바뀌면 과거 데이터로 배운 모델은 점점 안 맞게 됩니다.
> **코드가 고장 난 게 아니라 세상이 바뀐 것**입니다.

간단히 확인해 봅니다. 테스트 기간을 앞뒤로 나눠 성능을 비교합니다.

In [ ]:
# 테스트 기간을 절반으로 나눠 성능 변화를 봅니다.
te_sorted = test.sort_values("date")
중간 = len(te_sorted) // 2

전반 = te_sorted.iloc[:중간]
후반 = te_sorted.iloc[중간:]

print(f"{'구간':<8}{'기간':<26}{'행 수':>8}{'정확도':>10}")
print("-" * 54)
for 이름, 구간 in [("전반", 전반), ("후반", 후반)]:
    점수 = pipe.score(구간[FEATURES], 구간["target_up"])
    기간 = f"{구간.date.min().date()} ~ {구간.date.max().date()}"
    print(f"{이름:<8}{기간:<26}{len(구간):>8,}{점수:>10.4f}")

print()
print("학습 시점에서 멀어질수록 성능이 떨어지는지 확인합니다.")
print("(이 데이터는 기간이 짧아 차이가 작을 수 있습니다)")

### 대응

```
① 예측 결과를 기록해 둔다
② 실제 정답이 나오면 성능을 다시 측정한다
③ 기준 아래로 떨어지면 재학습한다
```

> 💡 **교안 19의 파이프라인이 그 자리를 맡습니다**
>
> 수집 → 정제 → 적재에 **재학습**을 붙이면 됩니다.
> 정기적으로 돌면서 모델을 갱신하는 구조입니다. 이것을 자동화한 것을 **MLOps** 라고 부릅니다.

---
## 정리

| 개념 | 핵심 |
|---|---|
| 왜 저장하나 | 학습은 느리고 예측은 빠르다 — **미리 해두고 꺼내 쓴다** |
| 저장 방법 | `joblib.dump` / `joblib.load` 두 줄 |
| **모델만 저장** | **스케일러가 없으면 조용히 틀린다** |
| 해결 | **Pipeline 통째로** 저장 |
| **피처 순서** | **numpy 로 넘기면 sklearn 도 못 막는다** |
| 함께 남길 것 | 피처 목록 · 학습 기간 · 성능 · 버전 |
| 로드 시점 | 요청마다 말고 **서버 시작 시 한 번** |
| 모델 수명 | 데이터 드리프트 — 정기적으로 재측정 |

---

### 직접 해보기

1. `models/pipeline.pkl` 을 **새 노트북에서** 로드해 예측해 보세요.
2. `meta.json` 을 텍스트 에디터로 열어보세요. `pkl` 과 무엇이 다른가요?
3. 피처를 하나 빼고 예측하면 어떤 에러가 나나요?

---

> 💡 **다음 단계**
>
> 저장한 모델을 **웹 서버에 올려 API 로 제공**하는 것이 다음 이야기입니다.
> 그때 이 노트북의 두 가지가 그대로 쓰입니다.
>
> - 서버가 뜰 때 한 번 로드하기
> - 요청으로 받은 JSON 을 **저장된 피처 순서대로** 정렬하기